# ParkVision - Experience B (YOLOv8s)
Notebook Colab de l'experience B. **Ne pas utiliser "Run all"** : executer les cellules dans l'ordre.
Etapes : 1 GPU, 2 clone de la branche `experience-b-yolov8s`, 3 donnees + DVC, 4 installation,
5 verification, 6 sauvegarde sur Drive, 7 essai court, 8 entrainement complet, 9 export MLflow,
10 analyse des erreurs. Aucun jeton n'est enregistre dans ce fichier.

## 1. GPU

In [ ]:
import sys
import torch

print("Python :", sys.version.split()[0])
print("PyTorch :", torch.__version__)
print("GPU disponible :", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Choisir Execution > Modifier le type d'execution > GPU (T4).")
print("GPU :", torch.cuda.get_device_name(0))

## 2. Cloner la branche experience-b-yolov8s (depot prive)

In [ ]:
import os
import subprocess
from getpass import getpass
from pathlib import Path

BRANCH = "experience-b-yolov8s"
project = Path("/content/ParkVision")

if project.exists():
    print("Le dossier existe deja : ne rien recloner.")
else:
    token = getpass("Jeton GitHub : ").strip()
    # Le jeton est transmis temporairement a Git, sans etre ecrit dans l'URL.
    helper = ('!f() { printf "%s\\n" "username=x-access-token" '
              '"password=$PARKVISION_GH_TOKEN"; }; f')
    result = subprocess.run(
        ["git", "-c", "credential.helper=", "-c", f"credential.helper={helper}",
         "clone", "--branch", BRANCH, "https://github.com/Nourchenne/ParkVision.git",
         str(project)],
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0", "PARKVISION_GH_TOKEN": token},
        capture_output=True, text=True,
    )
    del token
    print(result.stdout, result.stderr)
    result.check_returncode()

print(subprocess.check_output(["git", "branch", "--show-current"], cwd=project, text=True))
print(subprocess.check_output(["git", "log", "-1", "--oneline"], cwd=project, text=True))
assert (project / "configs/train_b.yaml").is_file(), "train_b.yaml introuvable : mauvaise branche."

## 3. Donnees brutes (ZIP du Drive partage) puis DVC

In [ ]:
%pip install -q --upgrade gdown

In [ ]:
import re
from pathlib import Path
from zipfile import ZipFile, is_zipfile

import gdown

link = input("Lien de partage (ou identifiant) du fichier ParkVision_raw.zip : ").strip()
match = re.search(r"/d/([\w-]+)|[?&]id=([\w-]+)", link)
file_id = (match.group(1) or match.group(2)) if match else link
archive = Path("/content/ParkVision_raw_drive.zip")

if not is_zipfile(archive):
    gdown.download(id=file_id, output=str(archive), quiet=False)
if not is_zipfile(archive):
    raise RuntimeError("ZIP non recupere : verifier le lien de partage et ses droits.")

print(f"Taille : {archive.stat().st_size / 1024**3:.2f} Go (attendu environ 1,66 Go)")
with ZipFile(archive) as zipped:
    damaged = zipped.testzip()
    if damaged:
        raise RuntimeError(f"Fichier endommage : {damaged}")
print("Archive verifiee.")

In [ ]:
from pathlib import Path
from zipfile import ZipFile

project = Path("/content/ParkVision")
with ZipFile("/content/ParkVision_raw_drive.zip") as zipped:
    zipped.extractall(project / "data")

expected = {"train": 8691, "valid": 2483, "test": 1242}
for split, count in expected.items():
    folder = project / "data" / "raw" / split
    images = {p.stem for p in (folder / "images").glob("*.jpg")}
    labels = {p.stem for p in (folder / "labels").glob("*.txt")}
    print(f"{split} : {len(images)} images | {len(labels)} annotations")
    if len(images) != count or images != labels:
        raise ValueError(f"Effectif ou correspondance incorrect : {split}")
print("Donnees brutes conformes (8691 / 2483 / 1242).")

In [ ]:
%pip install -q dvc==3.67.1

In [ ]:
import subprocess
import sys
import yaml
from pathlib import Path

project = Path("/content/ParkVision")
reference = project / "data/raw.dvc"
before = yaml.safe_load(reference.read_text())

def dvc(*args):
    result = subprocess.run([sys.executable, "-m", "dvc", *args], cwd=project,
                            input="n\n",  # refuser toute modification des donnees
                            capture_output=True, text=True)
    print(result.stdout, result.stderr)
    result.check_returncode()

dvc("commit", "data/raw.dvc")
if yaml.safe_load(reference.read_text()) != before:
    raise RuntimeError("La reference DVC a change : arreter et verifier.")
dvc("repro", "prepare_yolo")
dvc("status")
# Attendu : train 8613, valid 2500, test 1029 ; dvc.lock inchange.
print(subprocess.run(["git", "diff", "--stat", "--", "dvc.lock"], cwd=project,
                     capture_output=True, text=True).stdout or "dvc.lock inchange.")

## 4. Installer les versions du projet et verifier le GPU

In [ ]:
%cd /content/ParkVision
%pip install -q -r requirements-training.txt

In [ ]:
import torch, ultralytics, mlflow

print("PyTorch :", torch.__version__, "| Ultralytics :", ultralytics.__version__,
      "| MLflow :", mlflow.__version__)
with torch.no_grad():
    a = torch.randn(256, 256, device="cuda")
    assert torch.isfinite(a @ a.T).all().item()
del a
torch.cuda.empty_cache()
print("Calcul sur le GPU : reussi.")

## 5. Verifier la branche et les donnees

In [ ]:
%cd /content/ParkVision
!git branch --show-current
!git log -1 --oneline
!python src/models/train.py --config configs/train_b.yaml --check

## 6. Sauvegarde des poids sur Drive
Les sorties YOLO sont ecrites dans `Mon Drive/ParkVision_B/runs_detect/` via un lien symbolique.
Renseigner votre nom avant d'executer (il est enregistre dans le tag `member` de MLflow).

In [ ]:
import os
from pathlib import Path
from google.colab import drive

os.environ["PARKVISION_MEMBER"] = "Prenom Nom"  # A REMPLACER
drive.mount("/content/drive")

project = Path("/content/ParkVision")
destination = Path("/content/drive/MyDrive/ParkVision_B/runs_detect")
destination.mkdir(parents=True, exist_ok=True)

sorties = project / "runs" / "detect"
sorties.parent.mkdir(parents=True, exist_ok=True)
if sorties.is_symlink():
    assert sorties.resolve() == destination.resolve(), "Lien existant a verifier."
elif sorties.exists():
    raise RuntimeError("runs/detect existe deja : verifier son contenu avant de continuer.")
else:
    sorties.symlink_to(destination, target_is_directory=True)
print("Sorties YOLO ecrites dans :", sorties.resolve())

## 7. Essai court (condition de depart de l'entrainement long)

In [ ]:
%cd /content/ParkVision
!python -u src/models/train.py --config configs/train_b.yaml --smoke

In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive/ParkVision_B/runs_detect")
smoke = sorted(root.glob("B_yolov8s_baseline_smoke_*"), key=lambda p: p.stat().st_mtime)
assert smoke, "Dossier de l'essai court absent de Drive."
folder = smoke[-1]
for name in ("weights/last.pt", "weights/best.pt", "results.csv"):
    path = folder / name
    assert path.is_file(), f"Manquant sur Drive : {path}"
    print("OK", path, f"{path.stat().st_size / 1024**2:.1f} Mo")
print("Les poids de l'essai sont sur Drive : entrainement long autorise.")

## 8. Entrainement complet (une seule execution)

In [ ]:
%cd /content/ParkVision
!python -u src/models/train.py --config configs/train_b.yaml

## 9. Telecharger le ZIP MLflow du vrai entrainement

In [ ]:
from pathlib import Path
from google.colab import files

archive = Path(input("Chemin exact du ZIP affiche apres EXPORT A TELECHARGER : ").strip())
assert archive.is_file(), "Archive introuvable."
assert "B_yolov8s_baseline" in archive.name and "smoke" not in archive.name
files.download(str(archive))

## 10. Analyse des erreurs sur le lot valid
Utiliser les memes seuils que les experiences A et C (par defaut : confiance 0.25, NMS 0.7).
Les resultats sont ecrits dans `reports/experience_b/` puis copies sur Drive.

In [ ]:
import subprocess
import sys
from pathlib import Path

root = Path("/content/drive/MyDrive/ParkVision_B/runs_detect")
runs = [p for p in root.glob("B_yolov8s_baseline_*")
        if "smoke" not in p.name and (p / "weights/best.pt").is_file()]
assert runs, "Aucun best.pt de l'entrainement complet sur Drive."
best = max(runs, key=lambda p: p.stat().st_mtime) / "weights/best.pt"
print("Poids analyses :", best)

subprocess.run([sys.executable, "src/analysis/analyze_errors.py", "--weights", str(best),
                "--conf", "0.25", "--nms-iou", "0.7"], cwd="/content/ParkVision", check=True)

In [ ]:
import shutil

shutil.copytree("/content/ParkVision/reports/experience_b",
                "/content/drive/MyDrive/ParkVision_B/reports_experience_b", dirs_exist_ok=True)
print("Resultats copies sur Drive : ParkVision_B/reports_experience_b")